# Kalman vs Regression Lab Notebook

This notebook mirrors the Streamlit teaching flow: **Module A -> Module B -> Module C -> Module D**.

It then adds **Module E** as a roadmap extension for nonlinear filtering with EKF/UKF.

In [ ]:
import numpy as np
from IPython.display import Markdown, display

from kalman_regression_estimation_lab.demos import (
    module_a_static_vs_dynamic,
    module_b_same_data_different_lenses,
    module_c_ai_contexts,
    module_d_gnc_monte_carlo,
    module_e_nonlinear_ekf_ukf,
)
from kalman_regression_estimation_lab.plots import (
    fig_module_a_static,
    fig_module_a_dynamic,
    fig_module_a_velocity,
    fig_module_b_three_estimators,
    fig_module_b_predict_vs_update,
    fig_module_b_kalman_gain,
    fig_module_b_innovation,
    fig_module_b_rmse_bar,
    fig_module_d_2d_track,
    fig_module_d_velocity,
    fig_module_d_position_error,
    fig_module_e_track_comparison,
    fig_module_e_rmse,
)

display(Markdown("Environment ready."))

## Module A - Static vs Dynamic Estimation

- OLS fits a single global line.
- Kalman tracks state step-by-step and infers velocity from position measurements.

In [ ]:
data_a = module_a_static_vs_dynamic(
    static_noise_std=2.0,
    process_accel_std=0.2,
    measurement_std=2.0,
    seed=7,
)

fig_module_a_static(data_a["static"]).show()
fig_module_a_dynamic(data_a["dynamic"]).show()
fig_module_a_velocity(data_a["dynamic"]).show()

## Module B - Same Data, Three Estimators

This is the same visual panel set as the app:

1. Main overlay (measurements + OLS + RLS + Kalman)
2. Kalman predict vs update
3. Kalman gain
4. Innovation sequence
5. RMSE comparison

In [ ]:
data_b = module_b_same_data_different_lenses(
    process_accel_std=0.30,
    measurement_std=2.5,
    maneuver_step=85,
    maneuver_accel=1.0,
    seed=9,
)

fig_module_b_three_estimators(data_b).show()
fig_module_b_predict_vs_update(data_b).show()
fig_module_b_kalman_gain(data_b).show()
fig_module_b_innovation(data_b).show()
fig_module_b_rmse_bar(data_b).show()

## Module C - Algorithm Selection Guide

Use this to choose OLS, RLS, Kalman, or neural methods based on constraints and objective.

In [ ]:
ctx = module_c_ai_contexts()

display(Markdown("### Kalman is usually best for"))
for item in ctx["kalman_good_for"]:
    display(Markdown(f"- {item}"))

display(Markdown("### Regression or neural methods are enough for"))
for item in ctx["regression_or_nn_enough_for"]:
    display(Markdown(f"- {item}"))

display(Markdown("### Rule of thumb"))
for item in ctx["selection_rule_of_thumb"]:
    display(Markdown(f"- {item}"))

## Module D - GNC 2D Position and Velocity Tracking

The sensor observes x/y position with noise. Kalman reconstructs x/y/vx/vy with uncertainty-aware updates.

In [ ]:
data_d = module_d_gnc_monte_carlo(
    n_runs=40,
    process_accel_std=0.25,
    measurement_std=3.0,
    seed=21,
)

fig_module_d_2d_track(data_d).show()
fig_module_d_velocity(data_d).show()
fig_module_d_position_error(data_d).show()

## Module E - Roadmap Extension: Nonlinear EKF/UKF

This milestone adds nonlinear measurements from a radar-like sensor: **range + bearing**.

- EKF linearizes the measurement model with a Jacobian at each step.
- UKF avoids Jacobians and uses sigma points through the nonlinear transform.

In [ ]:
data_e = module_e_nonlinear_ekf_ukf(
    n_steps=140,
    dt=1.0,
    process_accel_std=0.25,
    range_std=2.5,
    bearing_std_rad=0.03,
    seed=77,
)

fig_module_e_track_comparison(data_e).show()
fig_module_e_rmse(data_e).show()

true_xy = data_e["true_state"][:, :2]
rmse_naive = float(np.sqrt(np.mean((data_e["measurements_cartesian_naive"] - true_xy) ** 2)))
rmse_ekf = float(np.sqrt(np.mean((data_e["ekf_estimate"][:, :2] - true_xy) ** 2)))
rmse_ukf = float(np.sqrt(np.mean((data_e["ukf_estimate"][:, :2] - true_xy) ** 2)))

display(Markdown(f"Naive RMSE: **{rmse_naive:.3f}**, EKF RMSE: **{rmse_ekf:.3f}**, UKF RMSE: **{rmse_ukf:.3f}**"))